In [1]:
%pwd

'd:\\Revision\\Covid-19_detection_with_MLOPS\\research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'd:\\Revision\\Covid-19_detection_with_MLOPS'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    train_dataset_path: Path
    val_dataset_path: Path
    params_image_size: list
    params_batch_size: int
    params_validation_split: float

In [5]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        params = self.params

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            train_dataset_path=Path(config.train_dataset_path),
            val_dataset_path=Path(config.val_dataset_path),
            params_image_size=params.IMAGE_SIZE,
            params_batch_size=params.BATCH_SIZE,
            params_validation_split=params.VALIDATION_SPLIT
        )

        return data_transformation_config

     

In [8]:
import os
import tensorflow as tf
from pathlib import Path
from cnnClassifier import logger


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def transform_and_save_data(self):
        logger.info(f"Loading and processing data from {self.config.data_path}")
        
        img_shape = tuple(self.config.params_image_size[:-1])

        # 1. Load, Resize, and Encode Labels
        train_ds = tf.keras.utils.image_dataset_from_directory(
            self.config.data_path,
            validation_split=self.config.params_validation_split,
            subset="training",
            seed=123,
            image_size=img_shape,
            batch_size=self.config.params_batch_size,
            label_mode='categorical'
        )
        
        val_ds = tf.keras.utils.image_dataset_from_directory(
            self.config.data_path,
            validation_split=self.config.params_validation_split,
            subset="validation",
            seed=123,
            image_size=img_shape,
            batch_size=self.config.params_batch_size,
            label_mode='categorical'
        )

        # 2. Normalization
        logger.info("Applying normalization transformation...")
        normalization_layer = tf.keras.layers.Rescaling(1./255)
        
        train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
        val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)

        # --- NEW LOGGING CODE ---
        # Log the shape specifications of the batches (Features and Labels)
        train_features_spec, train_labels_spec = train_ds.element_spec
        val_features_spec, val_labels_spec = val_ds.element_spec
        
        logger.info(f"Training Batch Image Shape: {train_features_spec.shape}")
        logger.info(f"Training Batch Label Shape: {train_labels_spec.shape}")
        logger.info(f"Validation Batch Image Shape: {val_features_spec.shape}")
        # ------------------------

        # 3. Optimize for Performance
        train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
        val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

        # 4. Save Transformed Datasets
        logger.info(f"Saving transformed datasets...")
        train_ds.save(str(self.config.train_dataset_path))
        val_ds.save(str(self.config.val_dataset_path))
        
        logger.info("Data Preprocessing and Transformation completed successfully.")

In [9]:
STAGE_NAME = "Data Transformation Stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.transform_and_save_data()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e

[2026-06-27 15:36:03,234: INFO: 3634220854: >>>>>> stage Data Transformation Stage started <<<<<<]
[2026-06-27 15:36:03,247: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-27 15:36:03,255: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-27 15:36:03,258: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-27 15:36:03,260: INFO: common: created directory at: artifacts]
[2026-06-27 15:36:03,261: INFO: common: created directory at: artifacts/data_transformation]
[2026-06-27 15:36:03,262: INFO: 1329139506: Loading and processing data from artifacts\data_ingestion\dataset]
Found 13808 files belonging to 2 classes.
Using 11047 files for training.
Found 13808 files belonging to 2 classes.
Using 2761 files for validation.
[2026-06-27 15:36:03,759: INFO: 1329139506: Applying normalization transformation...]
[2026-06-27 15:36:03,818: INFO: 1329139506: Training Batch Image Shape: (None, 224, 224, 3)]
[2026-06-27 15:36:03,819: INFO: 13291